# Isochrone Analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mihiarc/socialmapper/blob/main/docs/notebooks/02-isochrone-analysis.ipynb)

## Learning Objectives

By the end of this notebook, you will be able to:

- Understand what isochrones are and why they matter
- Create isochrones for different travel modes (walk, bike, drive)
- Compare accessibility across locations
- Work with isochrone geometry for spatial analysis

## Prerequisites

- Completed [01-Getting Started](01-getting-started.ipynb)

## Setup

In [ ]:
# Install SocialMapper from GitHub (latest version)
!pip install -q "socialmapper @ git+https://github.com/mihiarc/socialmapper.git"

# IMPORTANT: After install, go to Runtime > Restart session, then skip this cell

In [ ]:
import socialmapper
print(f"SocialMapper v{socialmapper.__version__}")

from socialmapper import create_isochrone
print("Ready!")

## What is an Isochrone?

An **isochrone** (from Greek: iso = equal, chronos = time) is a polygon that represents all locations reachable from a starting point within a given time limit.

Unlike simple radius-based buffers, isochrones account for:
- Road networks and paths
- Travel mode (walking, biking, driving)
- Terrain and obstacles

> **Tip:** Isochrones are essential for realistic accessibility analysis. A 15-minute drive covers much more area than a 15-minute walk, and neither is a simple circle!

In [ ]:
# Create a basic isochrone
isochrone = create_isochrone(
    location="Portland, OR",
    travel_time=15,
    travel_mode="drive"
)

# Examine the structure
print("Isochrone structure:")
print(f"  Type: {isochrone['type']}")
print(f"  Geometry type: {isochrone['geometry']['type']}")
print(f"\nProperties:")
for key, value in isochrone['properties'].items():
    print(f"  {key}: {value}")

## Travel Modes

SocialMapper supports three travel modes:

| Mode | Description | Typical Speed | Use Case |
|------|-------------|---------------|----------|
| `walk` | Pedestrian paths | ~5 km/h | Food deserts, transit access |
| `bike` | Cycling routes | ~15 km/h | Bike infrastructure studies |
| `drive` | Road network | ~30-50 km/h | Service area analysis |

In [ ]:
# Compare travel modes for the same location and time
location = "Portland, OR"
travel_time = 15

modes = ["walk", "bike", "drive"]
results = {}

print(f"15-minute travel from {location}:")
print("=" * 40)

for mode in modes:
    iso = create_isochrone(
        location=location,
        travel_time=travel_time,
        travel_mode=mode
    )
    area = iso['properties']['area_sq_km']
    results[mode] = area
    print(f"{mode.capitalize():8} {area:>8.2f} km²")

# Show relative sizes
print("\nRelative coverage:")
walk_area = results['walk']
print(f"  Biking covers {results['bike']/walk_area:.1f}x walking area")
print(f"  Driving covers {results['drive']/walk_area:.1f}x walking area")

## Routing Engine

SocialMapper uses the **Valhalla** routing engine for generating isochrones. Valhalla is a fast, open-source routing engine that uses OpenStreetMap data.

Key characteristics:
- **Fast**: Typical response time is 0.5-2 seconds
- **No API key required**: Uses the public OpenStreetMap Valhalla instance by default
- **All travel modes**: Supports walking, biking, and driving
- **Accurate**: Uses actual road network data from OpenStreetMap

> **Tip:** You can set a custom Valhalla endpoint via the `VALHALLA_URL` environment variable if you host your own instance.

## Location Input Methods

You can specify locations as:
1. **Place names** (geocoded automatically)
2. **Coordinates** (latitude, longitude)

In [ ]:
# Method 1: Place name
iso1 = create_isochrone(
    location="Pioneer Courthouse Square, Portland, OR",
    travel_time=10
)
print(f"Pioneer Courthouse Square: {iso1['properties']['area_sq_km']:.2f} km²")

# Method 2: Coordinates (lat, lon)
iso2 = create_isochrone(
    location=(45.5189, -122.6793),  # Same location by coords
    travel_time=10
)
print(f"Same location by coords: {iso2['properties']['area_sq_km']:.2f} km²")

## Multi-Time Analysis

Create isochrones for multiple travel times to show accessibility rings.

In [ ]:
# Create concentric isochrones
location = "Portland, OR"
times = [5, 10, 15, 20, 30]

print(f"Walking accessibility from {location}:")
print("=" * 40)

isochrones = []
for t in times:
    iso = create_isochrone(
        location=location,
        travel_time=t,
        travel_mode="walk"
    )
    isochrones.append(iso)
    area = iso['properties']['area_sq_km']
    print(f"{t:>3} min: {area:>6.2f} km²")

# Show how area grows with time
print("\nArea growth pattern:")
base_area = isochrones[0]['properties']['area_sq_km']
for i, t in enumerate(times):
    area = isochrones[i]['properties']['area_sq_km']
    multiplier = area / base_area
    print(f"  {t} min is {multiplier:.1f}x the 5-min area")

## Practical Example: Site Accessibility Comparison

Compare the accessibility of different potential locations for a new facility.

In [ ]:
# Compare potential sites in Portland
sites = [
    ("Downtown Portland", (45.5152, -122.6784)),
    ("Pearl District", (45.5268, -122.6836)),
    ("Alberta Arts District", (45.5590, -122.6475)),
]

print("Site Accessibility Analysis (15-min walk):")
print("=" * 50)

results = []
for name, coords in sites:
    iso = create_isochrone(
        location=coords,
        travel_time=15,
        travel_mode="walk"
    )
    area = iso['properties']['area_sq_km']
    results.append((name, area, iso))
    print(f"{name:25} {area:.2f} km²")

# Find best location
best = max(results, key=lambda x: x[1])
print(f"\nBest pedestrian accessibility: {best[0]} ({best[1]:.2f} km²)")

## Working with Isochrone Geometry

The isochrone result is GeoJSON, which you can use with other geographic tools.

In [ ]:
from shapely.geometry import shape

# Create an isochrone
iso = create_isochrone("Portland, OR", travel_time=15)

# Convert to Shapely geometry for spatial operations
polygon = shape(iso['geometry'])

print("Geometry properties:")
print(f"  Type: {polygon.geom_type}")
print(f"  Valid: {polygon.is_valid}")
print(f"  Area: {polygon.area:.6f} sq degrees")

# Get bounds
bounds = polygon.bounds
print(f"  Bounds: ({bounds[0]:.4f}, {bounds[1]:.4f}) to ({bounds[2]:.4f}, {bounds[3]:.4f})")

# Get centroid
centroid = polygon.centroid
print(f"  Centroid: ({centroid.y:.4f}, {centroid.x:.4f})")

## Checking Point Containment

Test whether specific locations fall within the isochrone.

In [ ]:
from shapely.geometry import shape, Point

# Create isochrone from downtown Portland
portland = (45.5152, -122.6784)
iso = create_isochrone(portland, travel_time=10, travel_mode="walk")
polygon = shape(iso['geometry'])

# Test some locations
test_locations = [
    ("Powell's Books", (45.5228, -122.6815)),
    ("Portland Art Museum", (45.5163, -122.6833)),
    ("OMSI", (45.5084, -122.6656)),
]

print("Locations within 10-min walk from downtown Portland:")
print("=" * 50)

for name, (lat, lon) in test_locations:
    point = Point(lon, lat)  # Note: Point takes (lon, lat)
    within = polygon.contains(point)
    status = "Yes" if within else "No"
    print(f"{name:30} {status}")

## Exporting Isochrones

Save isochrones for use in other applications.

In [ ]:
import json

# Create an isochrone
iso = create_isochrone("Portland, OR", travel_time=15)

# Save as GeoJSON
with open("portland_isochrone.geojson", "w") as f:
    json.dump(iso, f, indent=2)

print("Saved portland_isochrone.geojson")
print("File can be opened in QGIS, Mapbox, or any GeoJSON viewer")

## Exercise: Transportation Equity Analysis

Compare car vs. walking accessibility to understand transportation equity.

In [ ]:
# Analyze transportation equity
location = "Portland, OR"

# What can you reach in 30 minutes by different modes?
drive_30 = create_isochrone(location, travel_time=30, travel_mode="drive")
walk_30 = create_isochrone(location, travel_time=30, travel_mode="walk")

drive_area = drive_30['properties']['area_sq_km']
walk_area = walk_30['properties']['area_sq_km']

print(f"30-minute accessibility in {location}:")
print("=" * 40)
print(f"By car:     {drive_area:>8.2f} km²")
print(f"On foot:    {walk_area:>8.2f} km²")
print(f"")
print(f"Accessibility gap: {drive_area/walk_area:.0f}x")
print(f"\nPeople without cars can access {(walk_area/drive_area)*100:.1f}%")
print(f"of the area available to drivers.")

## Troubleshooting

### Common Issues and Solutions

| Issue | Cause | Solution |
|-------|-------|----------|
| `NetworkError` | Valhalla service unavailable | Check internet connection, try again later |
| Small/odd shaped isochrone | Poor OSM data coverage | Try a different location |
| `InvalidLocationError` | Location not found | Use coordinates or check spelling |
| Timeout errors | Large travel time or busy server | Reduce travel time, retry after a pause |

In [ ]:
from socialmapper import SocialMapperError, NetworkError, ValidationError

# Example: Handle errors gracefully
def create_isochrone_safe(location, travel_time):
    """Create isochrone with error handling."""
    try:
        return create_isochrone(location, travel_time)
    except NetworkError as e:
        print(f"Network error: {e}")
        print("Check your internet connection and try again.")
        return None
    except ValidationError as e:
        print(f"Validation error: {e}")
        return None

# Test the safe function
result = create_isochrone_safe("Portland, OR", 15)
if result:
    print(f"Success! Area: {result['properties']['area_sq_km']:.2f} km²")

## Next Steps

Continue with:

- **[Points of Interest](03-points-of-interest.ipynb)** - Find what's within your isochrones
- **[Census Data](04-census-data.ipynb)** - Analyze who lives in these areas
- **[Mapping](05-mapping-visualization.ipynb)** - Visualize your isochrones